In [1]:

! pip install unsloth datasets transformers

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 29.4/29.4 MB 55.7 MB/s  0:00:00 eta 0:00:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 12.3/12.3 MB 134.7 MB/s  0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 945.8/945.8 kB 58.3 MB/s  0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 43.1/43.1 MB 57.8 MB/s  0:00:006m0:00:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.8/2.8 MB 99.4 MB/s  0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.4/3.4 MB 123.7 MB/s  0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.3/3.3 MB 118.7 MB/s  0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.6/3.6 MB 131.6 MB/s  0:00:00
  Attempting uninstall: torchao
    Found existing installation: torchao 0.10.0
    Uninstalling torchao-0.10.0:
      Successfully uninstalled torchao-0.10.0━━━  0/12 [torchao]
  Attempting uninstall: transformers0m━━━━━━━━━━━━━━━━━━━━━━━  5/12 [xformers]]
    Found existing installation: transformers 5.20.2━━━━━━━━━━  5/12 [xformers]
    Uninstalling transformers-5.20.

In [2]:
import os
import pandas as pds
import unsloth
import torch
import transformers
import datasets
import gc
import json
import time
from pathlib import Path


🦥 Unsloth: Will patch your computer to enable 2x faster free finetuning.
Unsloth: xformers is installed but cannot load with torch 2.11.0+cu130 (it was built for 2.10.0+cu128), so PyTorch attention is used instead.
To install a matching xformers, run `unsloth install-kernels xformers`, or `pip uninstall xformers` to hide this. UNSLOTH_ENABLE_LOGGING=1 shows the full error.
🦥 Unsloth Zoo will now patch everything to make training faster!


In [3]:
from transformers import (
    AutoModelForCausalLM,
    AutoTokenizer,
    BitsAndBytesConfig,
)
from unsloth import FastLanguageModel, is_bf16_supported

print("done")

done


In [5]:
from google.colab import drive
DATA_DIR = drive.mount('/content/drive')

Mounted at /content/drive


In [8]:
from datasets import load_dataset


trn_data = "/content/drive/MyDrive/FINE-TUNE/DATASET/dataset_train.jsonl"
tst_data = "/content/drive/MyDrive/FINE-TUNE/DATASET/dataset_test.jsonl"
vld_data = "/content/drive/MyDrive/FINE-TUNE/DATASET/dataset_validation.jsonl"

data_files = {
    "train": trn_data,
    "validation": vld_data,
    "test": tst_data,
}
datasets = load_dataset("json", data_files=data_files)

print(datasets)

Generating train split: 0 examples [00:00, ? examples/s]

Generating validation split: 0 examples [00:00, ? examples/s]

Generating test split: 0 examples [00:00, ? examples/s]

DatasetDict({
    train: Dataset({
        features: ['messages', 'metadata'],
        num_rows: 4605
    })
    validation: Dataset({
        features: ['messages', 'metadata'],
        num_rows: 499
    })
    test: Dataset({
        features: ['messages', 'metadata'],
        num_rows: 499
    })
})


In [9]:

MODEL_NAME = "Qwen/Qwen3-4B-Instruct-2507"
MAX_SEQ_LENGTH= 2048


model, tokenizer = FastLanguageModel.from_pretrained(
    model_name=MODEL_NAME,
    max_seq_length = MAX_SEQ_LENGTH,
    dtype = None,
    load_in_4bit = True
)

==((====))==  Unsloth 2026.10.8: Fast Qwen3 patching. Transformers: 5.17.0.
   \\   /|    Tesla T4. Num GPUs = 1. Max memory: 14.563 GB. Platform: Linux.
O^O/ \_/ \    Torch: 2.11.0+cu130. CUDA: 7.5. CUDA Toolkit: 13.0. Triton: 3.6.0
\        /    Bfloat16 = FALSE. FA [Xformers = None. FA2 = False]
 "-____-"     Free license: http://github.com/unslothai/unsloth
Unsloth: Fast downloading is enabled - ignore downloading bars which are red colored!


Loading weights:   0%|          | 0/398 [00:00<?, ?it/s]

In [10]:
#  saving some examples from base model to compare it with fine-tuned one

from transformers import pipeline

# 1. Extract the 3 splits from your `datasets` DatasetDict
train_dataset = datasets["train"]
val_dataset   = datasets["validation"]
test_dataset  = datasets["test"]

# 2. Pick 5 unseen test rows for our Before vs. After comparison
eval_test_slice = test_dataset.select(range(5))

def get_answers(model, tokenizer, test_slice):
    FastLanguageModel.for_inference(model)
    answers = []

    for r in test_slice:
        # Pass conversation history excluding the final assistant target
        prompt_messages = [m for m in r["messages"] if m["role"] != "assistant"]

        inputs = tokenizer.apply_chat_template(
            prompt_messages,
            tokenize=True,
            add_generation_prompt=True,
            return_tensors="pt",
        ).to("cuda")

        with torch.inference_mode():
            outputs = model.generate(
                input_ids=inputs,
                max_new_tokens=256,
                temperature=0.2,
                do_sample=False,
                use_cache=True,
                pad_token_id=tokenizer.eos_token_id,
            )
        # Slice off prompt tokens to keep only the newly generated response
        response = tokenizer.decode(outputs[0][inputs.shape[1]:], skip_special_tokens=True)
        answers.append(response.strip())

    return answers

print("Generating Base Model baseline answers...")
base_model_answers = get_answers(model, tokenizer,eval_test_slice)
print(f"\nSample 1 Base Answer:\n{base_model_answers[0][:250]}...")

Generating Base Model baseline answers...


Both `max_new_tokens` (=256) and `max_length`(=262144) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
Both `max_new_tokens` (=256) and `max_length`(=262144) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
Both `max_new_tokens` (=256) and `max_length`(=262144) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
Both `max_new_tokens` (=256) and `max_length`(=262144) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_


Sample 1 Base Answer:
A **500 Internal Server Error** from an "unhandled SQLAlchemy lazy load outside session scope" typically occurs when you're trying to access a relationship or attribute on a SQLAlchemy model object **after the database session has been closed or is n...


In [11]:
#  Adding LoRA adapter
model = FastLanguageModel.get_peft_model(
    model,
    r=16,
    target_modules=[
        "q_proj",
        "k_proj",
        "v_proj",
        "o_proj",
        "gate_proj",
        "up_proj",
        "down_proj",
    ],
    lora_alpha = 16,
    lora_dropout=0,
    bias = "none",
    use_gradient_checkpointing = "unsloth",
    random_state = 42,
    use_rslora=False,
    loftq_config=None

)

Unsloth 2026.10.8 patched 36 layers with 36 QKV layers, 36 O layers and 36 MLP layers.


Unsloth: compiling each transformer block as one region (Qwen3DecoderLayer).


In [13]:
FastLanguageModel.for_training(model)
from transformers import DataCollatorForSeq2Seq

def format_chat_prompts(batch):
    input_ids_list, labels_list, attention_mask_list = [], [], []
    for convo in batch["messages"]:
        prompt_ids = tokenizer.apply_chat_template(
            convo[:-1],
            tokenize=True,
            add_generation_prompt=True,
        )
        full_ids = tokenizer.apply_chat_template(
            convo,
            tokenize=True,
            add_generation_prompt=False,
        )
        
        ids = full_ids[:MAX_SEQ_LENGTH]
        prompt_len = min(len(prompt_ids), len(ids))
        labels = [-100] * prompt_len + ids[prompt_len:]

        input_ids_list.append(ids)
        labels_list.append(labels)
        attention_mask_list.append([1] * len(ids))

    return {
        "input_ids": input_ids_list,
        "labels": labels_list,
        "attention_mask": attention_mask_list,
    }

train_dataset = train_dataset.map(
    format_chat_prompts,
    batched=True,
    remove_columns=train_dataset.column_names,
)
val_dataset = val_dataset.map(
    format_chat_prompts,
    batched=True,
    remove_columns=val_dataset.column_names,
)

row = train_dataset[0]
supervised_ids = [t for t, l in zip(row["input_ids"], row["labels"]) if l != -100]
assert len(supervised_ids) > 0, "Assistant span is empty — label masking is broken."

model.print_trainable_parameters()
print("\n--- SUPERVISED TOKENS ---")
print(tokenizer.decode(supervised_ids[:200]))


Map:   0%|          | 0/4605 [00:00<?, ? examples/s]

Map:   0%|          | 0/499 [00:00<?, ? examples/s]

trainable params: 33,030,144 || all params: 4,055,498,240 || trainable%: 0.8145

--- SUPERVISED TOKENS ---
<think>

</think>

Measure lag per partition, processing time, fetch rate, GC pauses. Scale partitions and consumer instances matching key ordering constraints. Increase fetch size, reduce poll interval overhead, batch DB writes. Optimize deserialization avoiding reflection, reuse objects. Add autoscaling on lag and CPU. Handle rebalances with cooperative protocol and warmup. Test with replayed peak traffic in staging. Alert on sustained lag exceeding SLO with runbook to scale or shed non-critical traffic. Document throughput per consumer, partition count rationale, and cost versus latency tradeoff. Finally validate changes with automated tests peer review and staging verification to ensure correctness maintainability performance and alignment with project standards requirements and long term operational needs for production.<|im_end|>



In [14]:
from trl import SFTTrainer, SFTConfig
from transformers.trainer_utils import get_last_checkpoint

CHECKPOINT_DIR = "/content/drive/MyDrive/FINE-TUNE/qwen3_4b_checkpoints"
os.makedirs(CHECKPOINT_DIR, exist_ok=True)

training_args = SFTConfig(
    output_dir = CHECKPOINT_DIR,
    max_seq_length = MAX_SEQ_LENGTH,      # 2048

    # 1. Your Tuned T4 Batching (Effective Batch Size = 6 * 2 = 12)
    packing = False,
    per_device_train_batch_size = 2,      
    gradient_accumulation_steps = 6,      
    dataloader_num_workers = 1,           
    dataloader_pin_memory = True,

    # 2. Learning Schedule & 8-Bit Optimizer
    num_train_epochs = 3,
    learning_rate = 1e-4,
    warmup_ratio = 0.05,
    lr_scheduler_type = "cosine",
    optim = "adamw_8bit",

    # 3. Precision (fp16 on Tesla T4)
    fp16 = not is_bf16_supported(),
    bf16 = is_bf16_supported(),

    # 4. Logging, Validation & Google Drive Checkpoints
    logging_steps = 5,
    eval_strategy = "steps",
    eval_steps = 50,
    save_strategy = "steps",
    save_steps = 50,
    save_total_limit = 2,
    seed = 42,
    report_to = "none",                       
    dataset_kwargs={"skip_prepare_dataset": True},
)

data_collator = DataCollatorForSeq2Seq(
    tokenizer=tokenizer,
    pad_to_multiple_of=8,
    return_tensors="pt",
    padding=True,
)

trainer = SFTTrainer(
    model = model,
    tokenizer = tokenizer,
    train_dataset = train_dataset,
    eval_dataset = val_dataset,
    args = training_args,
    data_collator=data_collator,
)


# Auto-resume from Google Drive if Colab ever disconnects
last_checkpoint = get_last_checkpoint(CHECKPOINT_DIR)
if last_checkpoint is not None:
    print(f"🔄 Resuming from checkpoint: {last_checkpoint}")
    trainer_stats = trainer.train(resume_from_checkpoint=last_checkpoint)
else:
    print("🚀 Starting fresh packed training run...")
    trainer_stats = trainer.train()

Unsloth: transformers renamed `warmup_ratio` to `warmup_steps`. Forwarding your value to `warmup_steps` - update your code when convenient. If you also passed `warmup_steps` as 0.1, that is its default here and cannot be distinguished from leaving it unset, so `warmup_ratio` was used; drop `warmup_ratio` to keep it.
🚀 Starting fresh packed training run...


The tokenizer has new PAD/BOS/EOS tokens that differ from the model config and generation config. The model config and generation config were aligned accordingly, being updated with the tokenizer's values. Updated tokens: {'bos_token_id': None}.
==((====))==  Unsloth - 2x faster free finetuning | Num GPUs used = 1
   \\   /|    Num examples = 4,605 | Num Epochs = 3 | Total steps = 1,152
O^O/ \_/ \    Batch size per device = 2 | Gradient accumulation steps = 6
\        /    Data Parallel GPUs = 1 | Total batch size (2 x 6 x 1) = 12
 "-____-"     Trainable parameters = 33,030,144 of 4,055,498,240 (0.81% trained)


Unsloth: Will smartly offload gradients to save VRAM!
Unsloth: Double buffering enabled (parallel H2D + compute) for backward pass.


Step,Training Loss,Validation Loss
50,2.155782,2.960671
100,1.728604,2.582352
150,1.748288,2.413237
200,1.570826,2.308259
250,1.492547,2.231817
300,1.540060,2.186710
350,1.429749,2.145939
400,1.533640,2.127987
450,1.533070,2.114342
500,1.439990,2.094554


Filter:   0%|          | 0/499 [00:00<?, ? examples/s]

Unsloth: Not an error, but `num_items_in_batch` could not be counted for Qwen3ForCausalLM's labels, so each micro-batch loss is a mean.
Using gradient accumulation will be very slightly less accurate.
Read more on gradient accumulation issues here: https://unsloth.ai/blog/gradient
Unsloth: Restored added_tokens_decoder metadata in /content/drive/MyDrive/FINE-TUNE/qwen3_4b_checkpoints/checkpoint-50/tokenizer_config.json.
Unsloth: Restored added_tokens_decoder metadata in /content/drive/MyDrive/FINE-TUNE/qwen3_4b_checkpoints/checkpoint-100/tokenizer_config.json.
Unsloth: Restored added_tokens_decoder metadata in /content/drive/MyDrive/FINE-TUNE/qwen3_4b_checkpoints/checkpoint-150/tokenizer_config.json.
Unsloth: Restored added_tokens_decoder metadata in /content/drive/MyDrive/FINE-TUNE/qwen3_4b_checkpoints/checkpoint-200/tokenizer_config.json.
Unsloth: Restored added_tokens_decoder metadata in /content/drive/MyDrive/FINE-TUNE/qwen3_4b_checkpoints/checkpoint-250/tokenizer_config.json.
Unsl

In [15]:
from IPython.display import display

# 1. Generate answers using the newly trained model (automatically switches to inference mode!)
print("Generating Fine-Tuned Model answers on the same 5 unseen test questions...")
fine_tuned_answers = get_answers(model, tokenizer,eval_test_slice)

# Dynamically locate user prompt (skips system message at index 0)
questions = [
    next((m["content"] for m in r["messages"] if m["role"] == "user"), "")
    for r in eval_test_slice
]

# Dynamically locate target assistant answer
target_ground_truth = [
    next((m["content"] for m in reversed(r["messages"]) if m["role"] == "assistant"), "")
    for r in eval_test_slice
]

# 3. Build a clean side-by-side comparison table (using `pds` from your imports!)
pds.set_option("display.max_colwidth", None)  # Prevents Pandas from cutting off long text with "..."

comparison_df = pds.DataFrame({
    "Question": questions,
    "Target Answer (Dataset)": target_ground_truth,
    "Base Model (Before)": base_model_answers,
    "Fine-Tuned (After)": fine_tuned_answers,
})

print("\n--- QUALITATIVE COMPARISON: BEFORE VS. AFTER ---")
display(comparison_df)

Both `max_new_tokens` (=256) and `max_length`(=262144) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


Generating Fine-Tuned Model answers on the same 5 unseen test questions...


Both `max_new_tokens` (=256) and `max_length`(=262144) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
Both `max_new_tokens` (=256) and `max_length`(=262144) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
Both `max_new_tokens` (=256) and `max_length`(=262144) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
Both `max_new_tokens` (=256) and `max_length`(=262144) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_


--- QUALITATIVE COMPARISON: BEFORE VS. AFTER ---


,Question,Target Answer (Dataset),Base Model (Before),Fine-Tuned (After)
0,Debug backend 500 error from unhandled SQLAlchemy lazy load outside session scope.,"DetachedInstanceError means accessing lazy relationship after session closed. Fix by eager loading with selectinload joinedload in query, or mapping to DTO inside session before return. Avoid lazy in async where greenlet issues arise. Add repository method returning fully loaded DTO, close session via dependency yield finally. Log query path, enable SQL echo in staging. Test accessing relationships after return expecting no IO, cover nested relations, empty collections. Profile query count with assert_num_queries, avoid N plus one. Document loading strategy per endpoint, when to split endpoints versus expand param, and clean architecture rule keeping ORM models in adapter only with domain entities detached and serializable.","A **500 Internal Server Error** from an ""unhandled SQLAlchemy lazy load outside session scope"" typically occurs when you're trying to access a relationship or attribute on a SQLAlchemy model object **after the database session has been closed or is no longer active**, and the object was loaded with **lazy loading** (e.g., `lazy='dynamic'`, `lazy='select'`, or `lazy='joined'`).\n\nThis is a common issue in Flask, FastAPI, or Django-based apps using SQLAlchemy ORM, especially when:\n\n- You fetch an object from the database (e.g., via `session.query(MyModel).first()`),\n- Then try to access a related object (e.g., `obj.related_field`) **outside the session scope**,\n- And the relationship is configured with lazy loading (like `lazy='dynamic'` or `lazy='select'`).\n\n---\n\n### 🔍 Root Cause\n\nSQLAlchemy lazy loading means that related objects are **not loaded immediately** when you fetch the main object. Instead, they are loaded **on-demand** (e.g., when you access the relationship). If you try to access that relationship **after the session has been closed or is not active**, SQLAlchemy will raise a `RuntimeError` or `Attribute","<think>\n\n</think>\n\nLazy load triggers query outside session causing ORM error. Fix with eager loading via selectinload or joinedload, or session scoped query. Example session.execute selectinload. Add try except logging stack trace. Test with fresh session. Avoid lazy load in API path. Validate fix with dedicated unit test and integration check covering edge cases. Log relevant metrics, monitor production dashboards continuously, document change clearly in pull request for reviewers and future maintenance work. Include reproducible script, versioned config, seeded runs, ablation results, error analysis table, and follow-up monitoring plan to ensure durable improvement. Confirm with peer review and merge."
1,Prevent prompt injection via third party webhook content in support bot.,"Treat webhook payload as untrusted data, never system instructions. Isolate with delimiters and instruct model to ignore embedded directives. Sanitize HTML and strip control tokens before insertion. Enforce allowlisted tool actions with human approval for sensitive ops. Log raw payloads for audit and run injection detectors. Test with known jailbreak strings and indirect instruction smuggling. Apply least privilege API scopes for bot actions. Add output validation blocking exfiltration patterns. For backends, version prompts and review changes. This defense in depth reduces risk of data theft or unauthorized actions. It acknowledges models cannot perfectly distinguish data from instructions, so architectural guardrails must constrain impact even when injections bypass filters.","To **prevent prompt injection via third-party webhook content in a support bot**, you must implement a multi-layered security strategy that focuses on input validation, context isolation, and safe prompt handling. Prompt injection occurs when an attacker manipulates the input to the AI model to alter its behavior (e.g., bypassing safety checks, executing unintended act

In [17]:
! pip install evaluate

import evaluate

print("\n--- CALCULATING QUANTITATIVE ROUGE SCORES ---")
rouge = evaluate.load("rouge")

# 1. Compute ROUGE for Base Model vs Target Dataset
base_rouge = rouge.compute(
    predictions=base_model_answers,
    references=target_ground_truth
)

# 2. Compute ROUGE for Fine-Tuned Model vs Target Dataset
ft_rouge = rouge.compute(
    predictions=fine_tuned_answers,
    references=target_ground_truth
)

# 3. Build a clean score comparison table using Pandas
score_df = pds.DataFrame({
    "Metric": ["ROUGE-1 (Keywords)", "ROUGE-2 (Phrases)", "ROUGE-L (Structure)"],
    "Base Model": [
        f"{base_rouge['rouge1'] * 100:.2f}%",
        f"{base_rouge['rouge2'] * 100:.2f}%",
        f"{base_rouge['rougeL'] * 100:.2f}%"
    ],
    "Fine-Tuned": [
        f"{ft_rouge['rouge1'] * 100:.2f}%",
        f"{ft_rouge['rouge2'] * 100:.2f}%",
        f"{ft_rouge['rougeL'] * 100:.2f}%"
    ],
    "Improvement": [
        f"{(ft_rouge['rouge1'] - base_rouge['rouge1']) * 100:+.2f}%",
        f"{(ft_rouge['rouge2'] - base_rouge['rouge2']) * 100:+.2f}%",
        f"{(ft_rouge['rougeL'] - base_rouge['rougeL']) * 100:+.2f}%",
    ]
})

display(score_df)

# saving adapters to the google drive so that we can use the adapters later
ADAPTER_SAVE_PATH = "/content/drive/MyDrive/FINE-TUNE/qwen3_4b_lora_adapter"

print(f"\nSaving final LoRA adapter weights to: {ADAPTER_SAVE_PATH}")
model.save_pretrained(ADAPTER_SAVE_PATH)
tokenizer.save_pretrained(ADAPTER_SAVE_PATH)
print("✅ Training pipeline complete! LoRA adapters saved securely to Google Drive.")


--- CALCULATING QUANTITATIVE ROUGE SCORES ---


,Metric,Base Model,Fine-Tuned,Improvement
0,ROUGE-1 (Keywords),21.25%,35.28%,+14.03%
1,ROUGE-2 (Phrases),1.73%,7.24%,+5.51%
2,ROUGE-L (Structure),9.53%,20.76%,+11.24%



Saving final LoRA adapter weights to: /content/drive/MyDrive/FINE-TUNE/qwen3_4b_lora_adapter


Unsloth: Restored added_tokens_decoder metadata in /content/drive/MyDrive/FINE-TUNE/qwen3_4b_lora_adapter/tokenizer_config.json.


✅ Training pipeline complete! LoRA adapters saved securely to Google Drive.


In [18]:
import shutil

# 1. Export to Colab's fast local SSD instead of Google Drive
LOCAL_GGUF_PATH = "/content/Qwen_PRIME4B_AI_gguf"

model.save_pretrained_gguf(
    LOCAL_GGUF_PATH,
    tokenizer,
    quantization_method = "q4_k_m"
)

# 2. Once finished, copy the finished GGUF folder safely to Google Drive
DRIVE_DEST = "/content/drive/MyDrive/FINE-TUNE/Qwen_PRIME4B_AI_gguf"
shutil.copytree(LOCAL_GGUF_PATH, DRIVE_DEST, dirs_exist_ok=True)
print("✅ GGUF successfully built locally and copied to Google Drive!")

Unsloth: Merging model weights to 16-bit format...


config.json:   0%|          | 0.00/1.57k [00:00<?, ?B/s]

model.safetensors.index.json:   0%|          | 0.00/32.9k [00:00<?, ?B/s]

Unsloth: Restored added_tokens_decoder metadata in /content/Qwen_PRIME4B_AI_gguf/tokenizer_config.json.


Found HuggingFace hub cache directory: /root/.cache/huggingface/hub


...

model-00002-of-00002.safetensors: downloading bytes:           |  0.00B            


Unsloth: Preparing safetensor model files: 100%|██████████| 2/2 [01:56<00:00, 58.13s/it]


Note: tokenizer.model not found (this is OK for non-SentencePiece models)


Unsloth: Merging weights into 16bit: 100%|██████████| 2/2 [02:46<00:00, 83.01s/it]


Unsloth: Merge process complete. Saved to `/content/Qwen_PRIME4B_AI_gguf`
Unsloth: Converting to GGUF format...
==((====))==  Unsloth: Conversion from HF to GGUF information
   \\   /|    [0] Installing llama.cpp might take 3 minutes.
O^O/ \_/ \    [1] Converting HF to GGUF f16 might take 3 minutes.
\        /    [2] Converting GGUF f16 to ['q4_k_m'] might take 10 minutes each.
 "-____-"     In total, you will have to wait at least 16 minutes.

Unsloth: Installing llama.cpp. This might take 3 minutes...
Unsloth: Installing prebuilt llama.cpp b11798-mix-f8a7950 (app-b11798-mix-f8a7950-linux-x64-cpu.tar.gz) - skipping compilation.
Unsloth: Preparing converter script...
Unsloth: [1] Converting model into f16 GGUF format.
This might take 3 minutes...
Unsloth: Initial conversion completed! Files: ['/content/Qwen_PRIME4B_AI_gguf_gguf/Qwen3-4B-Instruct-2507.F16.gguf']
Unsloth: [2] Converting GGUF f16 into q4_k_m. This might take 10 minutes...
Unsloth: Model files cleanup...
Unsloth: All GGUF 

In [19]:
import shutil

# Copy from the correct local folder where Unsloth saved the .gguf file
shutil.copytree("/content/Qwen_PRIME4B_AI_gguf_gguf", "/content/drive/MyDrive/FINE-TUNE/Qwen_PRIME4B_AI_gguf", dirs_exist_ok=True)
print("✅ GGUF model successfully copied to Google Drive!")

✅ GGUF model successfully copied to Google Drive!
